<a href="https://colab.research.google.com/github/kavita0704/web_app/blob/main/AI_Driven_Sign_Language_Translator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mediapipe==0.10.20 --quiet
!pip install opencv-python-headless==4.10.0.84 --quiet
!pip install numpy pandas scikit-learn joblib tqdm --quiet


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

root = "/content/drive/MyDrive"
for item in os.listdir(root):
    print(item)


Documents
Jk
Jk2
Untitled document (12).gdoc
Untitled document (11).gdoc
Untitled document.docx
Untitled document (10).gdoc
Colab Notebooks
112.jpg
,lll.jpg
Untitled document (9).gdoc
kkkkk_page-0001.jpg
Styrene Plant — Input ⇄ Output .gdoc
mba666 individual project.gdoc
core_resume_kavita.pdf
core
fee_recipt_placement.pdf
sde
This Kirana 2025 CPM India report.gdoc
Eco_friendly_plan_templet.gdoc
Untitled document (8).gdoc
Section 1.gdoc
make table how to put in ppt and in table just ri....gsheet
List of Countries by Sugarcane Production (1).csv
mba666_group_project.gdoc
Untitled document (7).gdoc
Here’s a 4–5 slide deck outline (ready to add to your “MBA Future Trends & Strategic Implementation” section) created by combining strategic insights from your mba666_group_project and mba.gdoc
placement.csv
Copy of placement.csv
Untitled document (6).gdoc
Main (1).cpp
Kavita_220512.gdoc
Untitled document (5).gdoc
Untitled document (4).gdoc
Untitled document (3).gdoc
Untitled spreadsheet.gshee

In [4]:
dataset_root = "/content/drive/MyDrive/sign_dataset"


In [5]:
import os

base = "/content/drive/MyDrive"

candidate = "sign_dataset"  # <-- change this
path = os.path.join(base, candidate)
print("Checking:", path)

for root, dirs, files in os.walk(path):
    print("ROOT:", root)
    print("  Subfolders:", dirs[:5])
    print("  Sample files:", files[:5])
    break  # just first level


Checking: /content/drive/MyDrive/sign_dataset
ROOT: /content/drive/MyDrive/sign_dataset
  Subfolders: ['hello', 'thanks', 'yes', 'no', 'iloveyou']
  Sample files: []


In [6]:
dataset_root = "THE_PATH_YOU_FOUND"
print("Using dataset_root:", dataset_root)


Using dataset_root: THE_PATH_YOU_FOUND


In [8]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

mp_hands = mp.solutions.hands

def extract_hand_landmarks_from_image(image_bgr):
    """
    Returns a flat list [x1,y1,z1,...,x21,y21,z21] if a hand is found,
    otherwise returns None.
    """
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.5
    ) as hands:
        results = hands.process(image_rgb)
        if not results.multi_hand_landmarks:
            return None
        hand_landmarks = results.multi_hand_landmarks[0]
        data = []
        for lm in hand_landmarks.landmark:
            data.extend([lm.x, lm.y, lm.z])
        return data

# 👇👇 SET THIS TO THE ONE YOU FOUND
dataset_root = "/content/drive/MyDrive/RealTimeObjectDetection/Tensorflow/workspace/images/collect_images"
   # CHANGE THIS

# Prepare CSV
columns = ['label'] + [f'{axis}{i}' for i in range(21) for axis in ['x', 'y', 'z']]
data_rows = []

labels = sorted([d for d in os.listdir(dataset_root)
                 if os.path.isdir(os.path.join(dataset_root, d))])

print("Found labels (subfolders):", labels)

for label in labels:
    label_dir = os.path.join(dataset_root, label)
    image_files = [f for f in os.listdir(label_dir)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    print(f"\nProcessing label: {label}, images: {len(image_files)}")

    for img_name in tqdm(image_files):
        img_path = os.path.join(label_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue

        landmarks = extract_hand_landmarks_from_image(img)
        if landmarks is not None:
            row = [label] + landmarks
            data_rows.append(row)

df = pd.DataFrame(data_rows, columns=columns)
print("Total valid samples (hand detected):", len(df))

csv_path = "/content/drive/MyDrive/sign_data_from_images.csv"
df.to_csv(csv_path, index=False)
print("Saved CSV to:", csv_path)
df.head()


Found labels (subfolders): ['hello', 'iloveyou', 'no', 'thanks', 'yes']

Processing label: hello, images: 15


100%|██████████| 15/15 [00:05<00:00,  2.51it/s]



Processing label: iloveyou, images: 16


100%|██████████| 16/16 [00:04<00:00,  3.23it/s]



Processing label: no, images: 15


...

100%|██████████| 19/19 [00:06<00:00,  2.81it/s]



Processing label: yes, images: 10


100%|██████████| 10/10 [00:03<00:00,  2.97it/s]

Total valid samples (hand detected): 72
Saved CSV to: /content/drive/MyDrive/sign_data_from_images.csv


,label,x0,y0,z0,x1,y1,z1,x2,y2,z2,...,z17,x18,y18,z18,x19,y19,z19,x20,y20,z20
0,hello,0.891490,0.679332,5.879669e-08,0.764374,0.750964,-0.025874,0.629737,0.750706,-0.029606,...,-0.027726,0.631251,0.231107,-0.047084,0.573211,0.182363,-0.056368,0.512394,0.145695,-0.064386
1,hello,0.892117,0.734928,3.469940e-08,0.779757,0.839248,-0.027857,0.625952,0.865729,-0.027836,...,-0.021796,0.558704,0.318252,-0.047344,0.486048,0.287877,-0.064172,0.412582,0.273227,-0.077207
2,hello,0.579024,0.782402,4.210895e-07,0.472648,0.785592,-0.024250,0.367457,0.717383,-0.030976,...,-0.042180,0.565665,0.289822,-0.063468,0.564262,0.205403,-0.076522,0.558700,0.130639,-0.085554
3,hello,0.320109,0.933560,6.390811e-07,0.405648,0.963336,-0.040242,0.502493,0.943654,-0.053198,...,-0.011248,0.410171,0.572586,-0.024366,0.430081,0.516767,-0.033519,0.453970,0.468823,-0.040120
4,hello,0.415274,0.764183,5.274226e-07,0.500290,0.812017,-0.029361,0.600222,0.803922,-0.033706,...,-0.001496,0.536372,0.427302,-0.009631,0.558227,0.379132,-0.014414,0.581570,0.339992,-0.017963


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

csv_path = "/content/drive/MyDrive/sign_data_from_images.csv"
model_path = "/content/drive/MyDrive/sign_model_from_images.pkl"

df = pd.read_csv(csv_path)
print("Data shape:", df.shape)
print("Labels:", df['label'].unique())

X = df.drop('label', axis=1).values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

print("Training model...")
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))

joblib.dump(clf, model_path)
print("Model saved at:", model_path)


Data shape: (72, 64)
Labels: ['hello' 'iloveyou' 'no' 'thanks' 'yes']
Training model...
Accuracy: 0.9333333333333333

Classification report:
               precision    recall  f1-score   support

       hello       1.00      1.00      1.00         3
    iloveyou       1.00      1.00      1.00         3
          no       0.75      1.00      0.86         3
      thanks       1.00      0.75      0.86         4
         yes       1.00      1.00      1.00         2

    accuracy                           0.93        15
   macro avg       0.95      0.95      0.94        15
weighted avg       0.95      0.93      0.93        15

Model saved at: /content/drive/MyDrive/sign_model_from_images.pkl
